In [ ]:
import pandas as pd

df_distance = pd.read_csv("Trips_by_Distance.csv")
df_full = pd.read_csv("Trips_Full Data.csv")

df_distance.head()

In [ ]:
# cleaning data

# check missing values
print("Missing values BEFORE cleaning:")
print(df_distance.isna().sum())

# convert date
df_distance['Date'] = pd.to_datetime(df_distance['Date'])

# drop unnecessary columns ONLY (not rows)
df_distance = df_distance.drop(columns=[
    'State FIPS', 
    'State Postal Code', 
    'County FIPS', 
    'County Name'
])

# final check
print("\nMissing values AFTER cleaning:")
print(df_distance.isna().sum())

# preview
df_distance.head()

In [ ]:
# filter out the national data 

# we only need the national data level
national_df = df_distance[df_distance['Level'] == 'National']

# quick preview
national_df.head()

In [ ]:
# group by week using date (correct method)

weekly_home = national_df.groupby(
    national_df['Date'].dt.isocalendar().week
)['Population Staying at Home'].mean()

# preview
weekly_home.head()

In [ ]:
# plot the average amount of people staying at home per week

import matplotlib.pyplot as plt

weekly_home.plot(kind='bar', figsize=(10,5))

plt.title('Average Population Staying at Home per Week')
plt.xlabel('Week')
plt.ylabel('Population Staying at Home')

plt.show()

In [ ]:
# check the average number of trips by distance

distance_cols = [
    'Number of Trips 1-3',
    'Number of Trips 3-5',
    'Number of Trips 5-10',
    'Number of Trips 10-25',
    'Number of Trips 25-50',
    'Number of Trips 50-100',
    'Number of Trips 100-250'
]

distance_summary = national_df[distance_cols].mean()

# preview
distance_summary

In [ ]:
# plot with commas instead of the scientific notation which was 1e8

import matplotlib.pyplot as plt
import matplotlib.ticker as ticker

ax = distance_summary.plot(kind='bar')

# format y axis with commas
ax.yaxis.set_major_formatter(ticker.StrMethodFormatter('{x:,.0f}'))

plt.title('Average Number of Trips by Distance')
plt.xlabel('Distance Category')
plt.ylabel('Average Number of Trips')

plt.show()

In [ ]:
# filter the data for highest number of trips

set1 = national_df[national_df['Number of Trips 10-25'] > 10000000]
set2 = national_df[national_df['Number of Trips 50-100'] > 10000000]

# check how many rows it leaves us with 
print(len(set1))
print(len(set2))

In [ ]:
# plot both sets so we can compare when they spike 

import matplotlib.pyplot as plt
import matplotlib.ticker as ticker

plt.scatter(set1['Date'], set1['Number of Trips 10-25'], label='10-25 miles')
plt.scatter(set2['Date'], set2['Number of Trips 50-100'], label='50-100 miles')

plt.title('Trips above 10 million comparison')
plt.xlabel('Date')
plt.ylabel('Number of Trips')

# format y axis with commas instead of the scientific notation
ax = plt.gca()
ax.yaxis.set_major_formatter(ticker.StrMethodFormatter('{x:,.0f}'))

plt.legend()

# preview the chart
plt.show()

In [10]:
# remove missing values ONLY for modelling since the modelling wont work WITH the missing values

model_data = df_distance[['Number of Trips 10-25', 'Number of Trips 5-10']].dropna()

# define x and y again
x = model_data['Number of Trips 10-25'].values.reshape(-1, 1)
y = model_data['Number of Trips 5-10'].values

In [25]:
# build the linear regression model using the clean data only ( without the missing values )

from sklearn.linear_model import LinearRegression
import numpy as np

# use the cleaned data ( kept getting error NaN, this solves that)
model_data = df_distance[['Number of Trips 10-25', 'Number of Trips 5-10']].dropna()

x = model_data['Number of Trips 10-25'].values.reshape(-1, 1)
y = model_data['Number of Trips 5-10'].values

# train the model
model = LinearRegression()
model.fit(x, y)

# make its predictions
y_pred = model.predict(x)

In [ ]:
# evaluate the model

from sklearn.metrics import mean_squared_error, r2_score
import numpy as np

rmse = np.sqrt(mean_squared_error(y, y_pred))
r2 = r2_score(y, y_pred)

# print the models results
print("RMSE:", rmse)
print("R2:", r2)

In [ ]:
# plot he regression model results

import matplotlib.pyplot as plt

plt.scatter(x, y, label='Actual data')
plt.plot(x, y_pred, color='red', label='Regression line')

plt.title('Linear Regression Model')
plt.xlabel('Trips 10-25 miles')
plt.ylabel('Trips 5-10 miles')
plt.legend()

# preview chart

plt.show()

In [ ]:
# create a visualisation for the the average mount of travellers by distance

import matplotlib.pyplot as plt

distance_summary.plot(kind='bar')

plt.title('Average Travellers by Distance')
plt.xlabel('Distance Range')
plt.ylabel('Average Number of Travellers')

# preview chart
plt.show()

In [ ]:
# measure the time using pandas - sequential so line by line

import time

start = time.time()

# simple operation (same as your analysis)
weekly_home = national_df.groupby('Week')['Population Staying at Home'].mean()

end = time.time()

# print the time result

print("Pandas time:", end - start)

In [ ]:
# measure time using dask - parallel so multiple cores working simulatenously

import dask.dataframe as dd
from dask.distributed import Client

# start dask client
client = Client(n_workers=4)

# convert pandas dataframe to dask
dask_df = dd.from_pandas(national_df, npartitions=4)

start = time.time()

weekly_home_dask = dask_df.groupby('Week')['Population Staying at Home'].mean().compute()

end = time.time()

# print the time result
print("Dask time:", end - start)
